In [ ]:
from google.colab import files
uploaded = files.upload()  # select all 4 CSVs at once

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, skew
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

label_names = {0: 'Normal', 1: 'Gas Lock', 2: 'Erosion', 3: 'Friction'}

STARTUP_TRIM    = 30   # discard first 30s (startup transient)
GAS_FAULT_START = 100  # gas lock fault begins at t=100s

# All 10 files — noisy sensor columns, multiple severities
file_label_map = {
    'normal_noisy.csv': 0,
    'normal_varA.csv':      0,
    'normal_varB.csv':      0,
    'gas_mild.csv':         1,
    'gas_moderate.csv':     1,
    'gas_severe.csv':       1,
    'erosion_mild.csv':     2,
    'erosion_moderate.csv': 2,
    'erosion_severe.csv':   2,
    'friction_mild.csv':    3,
    'friction_moderate.csv':3,
    'friction_severe.csv':  3,
}

# Use noisy sensor columns — not clean simulation outputs
SIGNAL_COLS = ['I_sensor', 'omega_sensor', 'Q_sensor',
               'H_sensor', 'vib_sensor', 'slip_sensor']

dfs = []
for filename, label in file_label_map.items():
    df = pd.read_csv(filename)

    # Rename time column if needed
    time_col = [c for c in df.columns if 'time' in c.lower()]
    if time_col:
        df = df.rename(columns={time_col[0]: 'time'})

    # Trim startup / pre-fault period
    if label == 1:
        df = df[df['time'] >= GAS_FAULT_START].reset_index(drop=True)
    else:
        df = df[df['time'] >= STARTUP_TRIM].reset_index(drop=True)

    df['label']      = label
    df['fault_name'] = label_names[label]
    df['severity']   = filename.replace('.csv','')  # track severity
    dfs.append(df)
    print(f"{filename:<30} label={label}  rows={len(df)}")

raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(raw)}")
print(f"\nClass distribution:")
print(raw['fault_name'].value_counts())

In [ ]:
# ── CELL 1b — Add stochastic sensor noise ─────────────────────
# Modelica is deterministic — real sensor noise must be added here
# Using Gaussian white noise scaled to realistic SNR per sensor:
#   I_sensor:     SNR ~40dB → std = signal_std * 0.01
#   omega_sensor: ±1 rad/s random jitter
#   Q_sensor:     ±0.0002 m3/s flowmeter quantization noise
#   H_sensor:     ±0.3m pressure transducer noise
#   vib_sensor:   high noise — accelerometers are noisy (SNR ~25dB)
#   slip_sensor:  derived quantity — moderate noise

np.random.seed(42)  # reproducible

noise_config = {
    #  column          absolute std    (physically motivated)
    'I_sensor':     0.15,    # CT clamp: ~1% of 14.7A
    'omega_sensor': 1.0,     # tachometer: ±1 rad/s
    'Q_sensor':     0.0002,  # flowmeter: ±2% of 0.010 m3/s
    'H_sensor':     0.3,     # pressure transducer: ±0.3m
    'vib_sensor':   0.05,    # accelerometer: high noise floor
    'slip_sensor':  0.001,   # slip estimate: small but present
}

print("Adding stochastic Gaussian noise to sensor signals...")
print(f"{'Signal':<20} {'Signal mean':>15} {'Noise std':>12} {'SNR (dB)':>10}")
print("-"*60)

for col, noise_std in noise_config.items():
    if col in raw.columns:
        signal_mean = raw[col].abs().mean()
        snr_db = 20 * np.log10(signal_mean / noise_std)

        # Add independent Gaussian noise to every sample
        raw[col] = raw[col] + np.random.normal(
                        loc=0.0,
                        scale=noise_std,
                        size=len(raw))

        print(f"{col:<20} {signal_mean:>15.4f} {noise_std:>12.4f} {snr_db:>10.1f}")

print("\nStochastic noise added ✓")
print("Each run will produce slightly different results — this is correct behaviour")

# Verify noise is visible — plot one signal before/after
# (run raw signals plot in Cell 2 to see the noise)

In [ ]:
# Print all columns so we can pick the right ones
print("All columns:", raw.columns.tolist())

# These are the sensor signals the MLSensor will read
# Adjust names here if your CSV columns are named differently
# NEW — noisy sensor columns from OpenModelica
SIGNAL_COLS = ['I_sensor', 'omega_sensor', 'Q_sensor',
               'H_sensor', 'vib_sensor', 'slip_sensor']

# Verify all exist
missing = [c for c in SIGNAL_COLS if c not in raw.columns]
if missing:
    print(f"WARNING - missing columns: {missing}")
    print("Available:", raw.columns.tolist())
else:
    print("All signal columns found ✓")

# Quick plot of each signal per fault class
fig, axes = plt.subplots(len(SIGNAL_COLS), 1, figsize=(14, 3*len(SIGNAL_COLS)))
for i, col in enumerate(SIGNAL_COLS):
    for label, group in raw.groupby('label'):
        axes[i].plot(group['time'].values,
                     group[col].values,
                     label=label_names[label], alpha=0.8)
    axes[i].set_ylabel(col)
    axes[i].legend(fontsize=7)
    axes[i].grid(True, alpha=0.3)
axes[0].set_title('Raw signals — all fault classes')
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig('raw_signals.png', dpi=150)
plt.show()
print("Plot saved as raw_signals.png")


In [ ]:
# ── CELL 3 — Feature extraction ───────────────────────────────
from scipy.stats import kurtosis, skew

def extract_features(window, signal_cols):
    features = {}
    for col in signal_cols:
        x = window[col].values
        rms = np.sqrt(np.mean(x**2))
        features[f'{col}_mean']  = np.mean(x)
        features[f'{col}_std']   = np.std(x)
        features[f'{col}_rms']   = rms
        features[f'{col}_peak']  = np.max(np.abs(x))
        features[f'{col}_kurt']  = kurtosis(x)
        features[f'{col}_skew']  = skew(x)
        features[f'{col}_crest'] = np.max(np.abs(x)) / (rms + 1e-9)

    I     = window['I_sensor'].values
    Q     = window['Q_sensor'].values
    H     = window['H_sensor'].values
    w     = window['omega_sensor'].values

    I_mean = np.mean(I)
    Q_mean = max(np.mean(Q), 1e-4)
    H_mean = max(np.mean(H), 0.1)
    w_mean = max(np.mean(w), 1.0)

    features['decoupling_IQ']    = I_mean / Q_mean
    features['decoupling_IH']    = I_mean / H_mean
    features['pump_eff_proxy']   = (Q_mean * H_mean) / (w_mean**2)
    features['speed_load_ratio'] = w_mean / I_mean

    return features


def build_feature_dataset(df, signal_cols, window_size=20, step_size=10):
    records = []
    for label, group in df.groupby('label'):
        group = group.reset_index(drop=True)
        n = len(group)
        for start in range(0, n - window_size, step_size):
            window = group.iloc[start:start + window_size]
            feats = extract_features(window, signal_cols)
            feats['label']      = label
            feats['fault_name'] = label_names[label]
            feats['time_start'] = window['time'].iloc[0]
            records.append(feats)

    feature_df = pd.DataFrame(records)
    print(f"Feature dataset: {len(feature_df)} windows")
    print(f"Features per window: {len(feature_df.columns) - 3}")
    print(f"\nClass distribution:")
    print(feature_df['fault_name'].value_counts())
    return feature_df


feature_df = build_feature_dataset(raw, SIGNAL_COLS,
                                    window_size=20,
                                    step_size=10)

# Sanity check — no exploding values
print("\nFeature value ranges:")
numeric = feature_df.drop(columns=['label','fault_name','time_start'])
print(numeric.describe().loc[['min','max']].T.sort_values('max', ascending=False).head(10))

In [ ]:
# ── Temporal train/test split ─────────────────────────────────
drop_cols = ['label', 'fault_name', 'time_start']

train_dfs = []
test_dfs  = []

print("Temporal split per fault class:")
print("-"*45)

for label in sorted(feature_df['label'].unique()):
    class_df = feature_df[feature_df['label'] == label].sort_values('time_start')
    n = len(class_df)
    split_idx = int(n * 0.70)
    train_dfs.append(class_df.iloc[:split_idx])
    test_dfs.append(class_df.iloc[split_idx:])
    print(f"  {label_names[label]:<12}: {split_idx} train | {n-split_idx} test windows")

train_df = pd.concat(train_dfs).reset_index(drop=True)
test_df  = pd.concat(test_dfs).reset_index(drop=True)

X_train = train_df.drop(columns=drop_cols).values
y_train = train_df['label'].values
X_test  = test_df.drop(columns=drop_cols).values
y_test  = test_df['label'].values

# Fit scaler on training data only
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

feature_names = train_df.drop(columns=drop_cols).columns.tolist()

print(f"\nTotal → Train: {len(X_train)} | Test: {len(X_test)}")

# ── Train classifier ──────────────────────────────────────────
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
clf.fit(X_train_sc, y_train)
y_pred = clf.predict(X_test_sc)

# ── Results ───────────────────────────────────────────────────
print("\n" + "="*50)
print("CLASSIFICATION REPORT — Temporal Split")
print("="*50)
print(classification_report(y_test, y_pred,
      target_names=[label_names[i] for i in range(4)]))

# ── Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[label_names[i] for i in range(4)],
            yticklabels=[label_names[i] for i in range(4)])
plt.title('Confusion Matrix — Temporal Split\n(trained on first 70%, tested on last 30% of each class)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix_temporal.png', dpi=150)
plt.show()

In [ ]:
# ── Prepare X and y ───────────────────────────────────────────
drop_cols = ['label', 'fault_name', 'time_start']
X = feature_df.drop(columns=drop_cols).values
y = feature_df['label'].values

# ── Scale features ────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

feature_names = feature_df.drop(columns=drop_cols).columns.tolist()

# ── Train/test split ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

# ── Random Forest classifier ──────────────────────────────────
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# ── Results ───────────────────────────────────────────────────
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred,
      target_names=[label_names[i] for i in range(4)]))

# ── Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[label_names[i] for i in range(4)],
            yticklabels=[label_names[i] for i in range(4)])
plt.title('Confusion Matrix — ESP Fault Classifier')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1][:20]  # top 20

plt.figure(figsize=(10, 6))
plt.bar(range(20), importances[indices], align='center')
plt.xticks(range(20),
           [feature_names[i] for i in indices],
           rotation=45, ha='right', fontsize=9)
plt.title('Top 20 Feature Importances — Random Forest')
plt.ylabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("\nTop 10 most important features:")
for i in indices[:10]:
    print(f"  {feature_names[i]:<35} {importances[i]:.4f}")

In [ ]:
import pickle

# Save scaler and model
with open('esp_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('esp_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

# Save feature names (needed for inference)
with open('esp_feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

# Download all outputs
files.download('esp_scaler.pkl')
files.download('esp_classifier.pkl')
files.download('esp_feature_names.pkl')
files.download('confusion_matrix.png')
files.download('feature_importance.png')

print("All files saved and downloaded ✓")
print(f"Model size: {len(pickle.dumps(clf))/1024:.1f} KB")
print(f"Scaler size: {len(pickle.dumps(scaler))/1024:.1f} KB")

In [ ]:
!pip install micromlgen -q

from micromlgen import port

# Port Random Forest to C
c_code = port(clf, classmap={
    0: 'Normal',
    1: 'GasLock',
    2: 'Erosion',
    3: 'Friction'
})

with open('esp_classifier.h', 'w') as f:
    f.write(c_code)

print(f"C code generated ✓")
print(f"Flash estimate: {len(c_code.encode())/1024:.1f} KB")
print(f"First 40 lines preview:")
for line in c_code.split('\n')[:40]:
    print(line)

In [ ]:
scaler_h = '#ifndef ESP_SCALER_H\n#define ESP_SCALER_H\n\n'
scaler_h += f'#define N_FEATURES {len(feature_names)}\n\n'

scaler_h += 'static const float FEATURE_MEANS[] = {\n'
for i, (name, val) in enumerate(zip(feature_names, scaler.mean_)):
    comma = ',' if i < len(feature_names)-1 else ''
    scaler_h += f'  {val:.8f}f{comma}  /* {name} */\n'
scaler_h += '};\n\n'

scaler_h += 'static const float FEATURE_STDS[] = {\n'
for i, (name, val) in enumerate(zip(feature_names, scaler.scale_)):
    comma = ',' if i < len(feature_names)-1 else ''
    scaler_h += f'  {val:.8f}f{comma}  /* {name} */\n'
scaler_h += '};\n\n'

scaler_h += '#endif /* ESP_SCALER_H */\n'

with open('esp_scaler.h', 'w') as f:
    f.write(scaler_h)

print(f"Scaler header generated ✓")
print(f"Size: {len(scaler_h.encode())/1024:.1f} KB")

In [ ]:
feature_h = '''#ifndef ESP_FEATURES_H
#define ESP_FEATURES_H

#include <math.h>
#include <string.h>

#define WINDOW_SIZE     20      /* timesteps per inference window */
#define N_SIGNALS       6       /* number of sensor channels     */
#define STEP_SIZE       10      /* window stride in timesteps    */

/* Signal channel indices */
#define CH_I_SENSOR     0
#define CH_OMEGA        1
#define CH_Q_SENSOR     2
#define CH_H_SENSOR     3
#define CH_VIB_SENSOR   4
#define CH_SLIP_SENSOR  5

/* Fault class labels */
#define CLASS_NORMAL    0
#define CLASS_GASLOCK   1
#define CLASS_EROSION   2
#define CLASS_FRICTION  3

static const char* CLASS_NAMES[] = {
    "NORMAL", "GAS_LOCK", "EROSION", "FRICTION"
};

/* ── Feature vector buffer ───────────────────────────────── */
static float feature_vector[N_FEATURES];

/* ── Helper: mean ────────────────────────────────────────── */
static float compute_mean(float* x, int n) {
    float sum = 0.0f;
    for (int i = 0; i < n; i++) sum += x[i];
    return sum / n;
}

/* ── Helper: std ─────────────────────────────────────────── */
static float compute_std(float* x, int n, float mean) {
    float sum = 0.0f;
    for (int i = 0; i < n; i++) sum += (x[i]-mean)*(x[i]-mean);
    return sqrtf(sum / n);
}

/* ── Helper: kurtosis ────────────────────────────────────── */
static float compute_kurtosis(float* x, int n, float mean, float std) {
    if (std < 1e-9f) return 0.0f;
    float sum = 0.0f;
    for (int i = 0; i < n; i++) {
        float z = (x[i] - mean) / std;
        sum += z * z * z * z;
    }
    return (sum / n) - 3.0f;  /* excess kurtosis */
}

/* ── Helper: skewness ────────────────────────────────────── */
static float compute_skewness(float* x, int n, float mean, float std) {
    if (std < 1e-9f) return 0.0f;
    float sum = 0.0f;
    for (int i = 0; i < n; i++) {
        float z = (x[i] - mean) / std;
        sum += z * z * z;
    }
    return sum / n;
}

/* ── Helper: peak ────────────────────────────────────────── */
static float compute_peak(float* x, int n) {
    float peak = 0.0f;
    for (int i = 0; i < n; i++) {
        float ax = fabsf(x[i]);
        if (ax > peak) peak = ax;
    }
    return peak;
}

/*
 * extract_features()
 * Fills feature_vector[] from a WINDOW_SIZE x N_SIGNALS buffer.
 * window[sample][channel] layout.
 * Returns pointer to feature_vector for convenience.
 */
static float* extract_features(float window[WINDOW_SIZE][N_SIGNALS]) {

    int feat_idx = 0;

    /* ── Per-signal statistical features ─────────────────── */
    for (int ch = 0; ch < N_SIGNALS; ch++) {
        float x[WINDOW_SIZE];
        for (int t = 0; t < WINDOW_SIZE; t++) x[t] = window[t][ch];

        float mean  = compute_mean(x, WINDOW_SIZE);
        float std   = compute_std(x, WINDOW_SIZE, mean);
        float rms   = sqrtf(compute_mean((float[]){0}, 1)); /* computed below */

        /* RMS */
        float sum_sq = 0.0f;
        for (int t = 0; t < WINDOW_SIZE; t++) sum_sq += x[t]*x[t];
        rms = sqrtf(sum_sq / WINDOW_SIZE);

        float peak   = compute_peak(x, WINDOW_SIZE);
        float kurt   = compute_kurtosis(x, WINDOW_SIZE, mean, std);
        float skew   = compute_skewness(x, WINDOW_SIZE, mean, std);
        float crest  = (rms > 1e-9f) ? peak / rms : 0.0f;

        feature_vector[feat_idx++] = mean;
        feature_vector[feat_idx++] = std;
        feature_vector[feat_idx++] = rms;
        feature_vector[feat_idx++] = peak;
        feature_vector[feat_idx++] = kurt;
        feature_vector[feat_idx++] = skew;
        feature_vector[feat_idx++] = crest;
    }

    /* ── Cross-modal decoupling features ──────────────────── */
    float I_mean = feature_vector[0];  /* CH_I_SENSOR mean     */
    float w_mean = feature_vector[7];  /* CH_OMEGA mean        */
    float Q_mean = feature_vector[14]; /* CH_Q_SENSOR mean     */
    float H_mean = feature_vector[21]; /* CH_H_SENSOR mean     */

    if (Q_mean < 1e-4f) Q_mean = 1e-4f;
    if (H_mean < 0.1f)  H_mean = 0.1f;
    if (w_mean < 1.0f)  w_mean = 1.0f;

    feature_vector[feat_idx++] = I_mean / Q_mean;               /* decoupling_IQ    */
    feature_vector[feat_idx++] = I_mean / H_mean;               /* decoupling_IH    */
    feature_vector[feat_idx++] = (Q_mean * H_mean) / (w_mean * w_mean); /* pump_eff_proxy  */
    feature_vector[feat_idx++] = w_mean / I_mean;               /* speed_load_ratio */

    return feature_vector;
}

/*
 * normalize_features()
 * Applies StandardScaler: (x - mean) / std
 * Must be called on feature_vector before predict().
 */
#include "esp_scaler.h"
static void normalize_features(float* fv, int n) {
    for (int i = 0; i < n; i++) {
        fv[i] = (fv[i] - FEATURE_MEANS[i]) / (FEATURE_STDS[i] + 1e-9f);
    }
}

#endif /* ESP_FEATURES_H */
'''

with open('esp_features.h', 'w') as f:
    f.write(feature_h)

print("Feature extractor header generated ✓")

In [ ]:
from google.colab import files

files.download('esp_classifier.h')
files.download('esp_scaler.h')
files.download('esp_features.h')

# Print total flash estimate
total_bytes = (len(open('esp_classifier.h').read()) +
               len(open('esp_scaler.h').read()) +
               len(open('esp_features.h').read()))
print(f"\nTotal C code size: {total_bytes/1024:.1f} KB")
print(f"STM32F4 Flash:     512 KB available")
print(f"Headroom:          {(512*1024 - total_bytes)/1024:.1f} KB remaining")

In [ ]:
# Check what the generated classifier looks like
with open('esp_classifier.h', 'r') as f:
    content = f.read()

lines = content.split('\n')
print(f"Total lines: {len(lines)}")
print(f"Total size: {len(content.encode())/1024:.1f} KB")

# Find the predict function signature
for i, line in enumerate(lines):
    if 'predict' in line.lower() and '(' in line:
        print(f"\nLine {i}: {line}")

# Find class definitions
for i, line in enumerate(lines):
    if 'Normal' in line or 'GasLock' in line:
        print(f"Line {i}: {line}")
        break

In [ ]:
with open('esp_classifier.h', 'r') as f:
    lines = f.readlines()

# Print first 20 lines to see the structure
print("=== First 20 lines ===")
for i, line in enumerate(lines[:20]):
    print(f"{i+1}: {line}", end='')

# Find predict function
print("\n=== predict() definition ===")
for i, line in enumerate(lines):
    if 'predict' in line and ('int' in line or 'class' in line or 'Classifier' in line):
        print(f"{i+1}: {line}", end='')

In [ ]:
# Export representative test vectors for each class
# Takes the median window from each class — most representative sample
WINDOW_SIZE = 20
print("// ── Representative test vectors from training data ──")
print("// Copy these into MLSensor.ino to replace the hardcoded values\n")

for label in range(4):
    class_windows = feature_df[feature_df['label'] == label]

    # Get median window index — most representative of the class
    median_idx = len(class_windows) // 2
    sample = class_windows.iloc[median_idx]

    # Find the original raw window this feature row came from
    t_start = sample['time_start']
    class_raw = raw[raw['label'] == label].reset_index(drop=True)

    # Find matching timestep in raw data
    time_match = class_raw[class_raw['time'] >= t_start].head(WINDOW_SIZE)

    if len(time_match) < WINDOW_SIZE:
        print(f"// WARNING: not enough rows for {label_names[label]}")
        continue

    print(f"void load_{label_names[label].lower().replace(' ','_')}(void) {{")
    print(f"    // Source: simulation t={t_start:.1f}s, label={label_names[label]}")
    print(f"    float data[WINDOW_SIZE][N_SIGNALS] = {{")

    for t in range(WINDOW_SIZE):
        row = time_match.iloc[t]
        vals = [f"{row[col]:.6f}f" for col in SIGNAL_COLS]
        comma = ',' if t < WINDOW_SIZE - 1 else ''
        print(f"        {{{', '.join(vals)}}}{comma}")

    print(f"    }};")
    print(f"    memcpy(sensor_window, data, sizeof(data));")
    print(f"}}\n")